# 🛡️ FraudGuard — Google Colab Runner

Real-time credit-card fraud detection. Run all cells top-to-bottom.

**EN:** This notebook clones the repo, installs dependencies, downloads the dataset, trains the models, and launches the bilingual (English/Uzbek) Gradio UI with a public link.

**UZ:** Ushbu daftar repozitoriyani klon qiladi, kutubxonalarni o'rnatadi, ma'lumotlar to'plamini yuklab oladi, modellarni o'qitadi va ikki tilli (ingliz/o'zbek) Gradio interfeysini common havola bilan ishga tushiradi.

> Tip: Set Runtime → Change runtime type → CPU is enough (GPU not required).

## 1. Clone the repository / Repozitoriyani klon qilish

In [ ]:
import os
if not os.path.exists('FraudGuard'):
    !git clone https://github.com/MYS707team/FraudGuard.git
%cd FraudGuard
!ls -la

## 2. Install dependencies / Kutubxonalarni o'rnatish

In [ ]:
!pip install -q -r requirements.txt
print('Dependencies installed / Kutubxonalar o\u2018rnatildi')

## 3. Get the dataset / Ma'lumotlar to'plamini olish

**Option A — Kaggle API (recommended).** Upload your `kaggle.json` (Kaggle → Account → Create New API Token) when prompted.

**Option B — Manual upload.** Skip this cell and run the next one to upload `creditcard.csv` directly.

In [ ]:
# Option A: Kaggle API
from google.colab import files
import os, json

print('Upload your kaggle.json / kaggle.json faylini yuklang...')
uploaded = files.upload()
if 'kaggle.json' in uploaded:
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'wb') as f:
        f.write(uploaded['kaggle.json'])
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    !pip install -q kaggle
    !kaggle datasets download -d mlg-ulb/creditcardfraud -p data/raw
    !unzip -o data/raw/creditcardfraud.zip -d data/raw
    print('Dataset downloaded to data/raw/creditcard.csv')
else:
    print('No kaggle.json uploaded — use Option B (next cell).')

In [ ]:
# Option B: Manual upload of creditcard.csv (run only if you skipped Option A)
from google.colab import files
import os, shutil

if not os.path.exists('data/raw/creditcard.csv'):
    os.makedirs('data/raw', exist_ok=True)
    print('Upload creditcard.csv / creditcard.csv faylini yuklang...')
    up = files.upload()
    for name in up:
        if name.endswith('.csv'):
            shutil.move(name, 'data/raw/creditcard.csv')
    print('Saved to data/raw/creditcard.csv')
else:
    print('Dataset already present — skipping.')

## 4. Train the models / Modellarni o'qitish

Trains Logistic Regression, Random Forest, XGBoost (+SMOTE) and Isolation Forest, then picks the best by PR-AUC.

In [ ]:
!python -m src.train

## 5. Launch API + bilingual UI / API va ikki tilli interfeysni ishga tushirish

Starts the FastAPI backend in the background, then launches the Gradio UI. A public `*.gradio.live` link will appear below.

In [ ]:
import subprocess, time, os, requests, sys

# Start FastAPI backend in the background
api_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'api.main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

# Wait until /health responds
for _ in range(30):
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.ok:
            print('API ready:', r.json())
            break
    except Exception:
        time.sleep(1)
else:
    print('API did not start in time — check logs.')

In [ ]:
# Launch the Gradio UI with a public share link
os.environ['FRAUDGUARD_API_URL'] = 'http://localhost:8000'
from ui.app import build_ui
build_ui().launch(share=True)